# Custom encoder workflow

**Tier:** Docs-CI executable (CPU) · **Expected runtime:** under 10 seconds · **Hardware:** CPU

An encoder consumes a sequence observation and returns a fixed-size feature vector used by the policy. We create a minimal PyTorch module, check its shape, and map the steps to the extension contract.

In [ ]:
import torch
from torch import nn


class TinySequenceEncoder(nn.Module):
    def __init__(self, feature_dim: int, output_dim: int) -> None:
        """Initialize the illustrative projection encoder."""
        super().__init__()
        self.feature_dim = feature_dim
        self.output_dim = output_dim
        self.projection = nn.Linear(feature_dim, output_dim)

    def forward(self, observation: torch.Tensor) -> torch.Tensor:
        """Project the last observation row into a fixed-width embedding."""
        # observation: (batch, time, feature); summarize the latest visible row.
        return torch.tanh(self.projection(observation[:, -1, :]))


encoder = TinySequenceEncoder(feature_dim=8, output_dim=16)
observation = torch.zeros((2, 20, 8), dtype=torch.float32)
embedding = encoder(observation)
print(
    "input shape:",
    tuple(observation.shape),
    "feature dim:",
    encoder.feature_dim,
    "output shape:",
    tuple(embedding.shape),
)
assert embedding.shape == (2, 16)

## Integration checklist

1. Preserve the declared observation shape and normalization contract.
2. Register the architecture with a stable identifier.
3. Ensure the policy sees the configured output dimension.
4. Add shape, determinism, serialization, and checkpoint compatibility tests.
5. Resolve the architecture through configuration and verify with a small CPU smoke.

This minimal tensor example is illustrative; it is not registered as a production encoder. See [add an encoder](../../../how-to/add_encoder), [encoder implementations](../../../api/domains/models), and [model architecture reference](../../../reference/model_architectures).